# Leidenfrost Testcase
This is a testcase for evaporation. It is derived from the "film boiling" testcase.  
There a thin vapor film is situated above a heated wall.  
The flat liquid-vapor interface is disturbed according to the largest rayleigh-taylor instability wavelength.  
As the liquid evaporated a vapor bubble is forming and eventually emerging from the film.  
references for this testcase (in some points they possess some discrepancies)  
`10.1016/j.jcp.2006.07.035` (G)  
`10.1006/jcpn.200.6481` (WW)     
Here instead of a vapor film a liquid droplet is positioned above the heated wall.  

Note the following:
* We take the case with $\Delta T = 5K$
* We use the value for $h_v$ given in (WW)

In [2]:
//#r "..\..\binaries\BoSSSpad.dll"
//#r "BoSSSpadHFSI\BoSSSpad.dll"
#r "d:\Users\miao\Documents\JupyterNotebook\BoSSSpadMergeCFL\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
//using BoSSS.Application.XNSFE_Solver;
using HFSISolver;
using HFSISolver.SolidPhase;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using Microsoft.AspNetCore.Html;
Init();

In [3]:
string proj = "HFSI-Leidenfrost-Angle";

In [4]:
static var myBatch = ExecutionQueues[4];
static var myDb = myBatch.CreateOrOpenCompatibleDatabase("HFSI-Leidenfrost-Angle-3d");

In [5]:
BoSSSshell.WorkflowMgm.Init("HFSI-Leidenfrost-Angle-3d");

Project name is set to 'HFSI-Leidenfrost-Angle-3d'.
Default Execution queue is chosen for the database.
Opening existing database '\\dc3\userspace\miao\cluster\HFSI-Leidenfrost-Angle-3d'.


In [6]:
BoSSSshell.WorkflowMgm.DefaultDatabase = myDb;

Opening existing database '\\fdygitrunner\BoSSStests\Miao-Temp\HFSI-Leidenfrost-Angle-3d'.


In [6]:
//string proj = "XNSFE-Leidenfrost-Test-1";
//BoSSSshell.WorkflowMgm.Init(proj);
//BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

In [7]:
//var sessions = BoSSSshell.WorkflowMgm.DefaultDatabase.Sessions;
//sessions

## Clean up old simulations

In [8]:
// #r "..\binaries\ValidationTestRunner.dll"

In [9]:
// ValidationTestRunner.ValidationTestRunnerMain.DeleteDatabaseAndDeploymentsWhenOld(
//                "XNSFE-FilmBoiling",
//                "XNSFE-FilmBoiling*",
//                "delete_XNSFE-FilmBoiling", new TimeSpan(days: 25, hours: 1, minutes: 0, seconds: 1));

## Grid and Boundary/Initial Value configuration

In [7]:
static class Constants {
    // careful order of declaration matters!!!
    // material parameters
    public static double rho_A = 200.0;
    public static double rho_B = 5.0;

    public static double mu_A = 0.3;
    public static double mu_B = 0.005;

    public static double Sigma = 0.1;

    public static double c_A = 400.0;
    public static double c_B = 200.0;

    public static double k_A = 40.0;
    public static double k_B = 1.0;

    public static double hVap = 10000;
    public static double T_sat = 0; // 500K, shifted
    public static double T_wall = T_sat + 5.0;

    public static double alpha_A = k_A / (c_A * rho_A);
    public static double alpha_B = k_B / (c_B * rho_B);
    public static double eps = 1.0 - rho_B / rho_A;

    public static double g = 9.81;
    //public static double g = 9.81 * Math.Sqrt(2.0) / 2.0;

    public static double lambda = 2 * Math.PI * Math.Sqrt((3 * Sigma) / (g * Math.Abs(rho_A - rho_B)));
    public static double L = lambda / 2.0;

    public static double ST = 0.995 * L * 3.0 / 4.0; // 0.01; Solid Thickness
    public static double X0 = 0 * L;
    public static double Y0 = L * (1.0 / 4.0 + 1.0 / 16.0) + ST;
    public static double R = L / 4.0;

    // capillary timestep, for finest res + highest degree, use this, for comparability?! 
    // Is very small 1e-7 => 1e5 - 1e6 timesteps necessary => artificial surface tension?!
    public static Func<int, int, double> dt = (res, p) =>
        0.5 * Math.Sqrt((rho_A + rho_B) * Math.Pow(L / ((double)res * ((double)p + 1)), 3) / (2 * Math.PI * Math.Abs(Sigma)));
}

In [16]:
Constants.ST

0.0293591200521904

In [8]:
Constants.L

0.03934220442504576

In [12]:
0.015 / Math.Sqrt((Constants.Sigma / (Constants.rho_A * Constants.g))) //Sightly wrong Dimensionlessly maximum radius https://doi.org/10.1017/jfm.2022.66

2.1010711553871753

In [13]:
Constants.lambda

0.07868440885009152

In [14]:
Constants.rho_A * 1.5 *1.5 * Constants.R / Constants.Sigma

44.259979978176474

In [18]:
(Constants.k_B*Constants.mu_B*5)/(Constants.Sigma*Constants.hVap*Constants.rho_B*Math.Sqrt((Constants.Sigma / (Constants.rho_A * Constants.g)))) //The (dimensionless) evaporation number

0.0007003570517957251

In (WW) $h_v = 10^4$ in (G) $h_v = 10^3$ which is right?  
Form the plots in (G) we approximate  
$\Delta V(t = 0.455) \approx \Pi * 0.02^2 \approx 1.257*10^{-3}$  
Further:  
$\Delta V(t = 0.455) \approx \frac{\Delta m}{rho_v}$  
$\Delta m(t) \approx \frac{k_v \frac{\Delta T}{\delta}}{h_v}l_{\mathfrak{J}} t \approx 6.283*10^{-3}$
$h_v \approx \frac{k_v \frac{\Delta T}{\delta}}{\Delta m}l_{\mathfrak{J}} 0.455 \approx 11500$  
where:  
$l_{\mathfrak{J}} \approx \lambda$  
$\delta \approx 4/128 \lambda$  
We conclude, that $h_v = 10^4$ has to be used!  

In [16]:
static class GridFactory{
    public static GridCommons Grid(int res, IDatabaseInfo myDb){
        double h = Constants.L/res;
        double[] Xnodes = GenericBlas.Linspace(-Constants.L, Constants.L, 2 *res + 1);
        double[] Ynodes = GenericBlas.Linspace(0, 6 * Constants.L, 6*res + 1);

        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes);

        grd.EdgeTagNames.Add(1, "freeslip_ZeroGradient");
        grd.EdgeTagNames.Add(2, "pressure_outlet_ConstantTemperature");
        grd.EdgeTagNames.Add(3, "wall_ConstantTemperature");
        
        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 1;
            if((Math.Abs(X[0] - Xnodes.First()) < 1e-6) || (Math.Abs(X[0]- Xnodes.Last()) < 1e-6))
                return 1;
            else if((Math.Abs(X[1] - Ynodes.Last()) < 1e-6) )
                return 2;
            else if((Math.Abs(X[1] - Ynodes.First()) < 1e-6) )
                return 3;
            return et;
        });

        myDb.Controller.DBDriver.SaveGrid(grd, myDb);

        return grd;
    }

    public static GridCommons GridConvergence(int res, IDatabaseInfo myDb){
        double[] Xnodes = GenericBlas.Linspace(- Constants.L, 11.0 * Constants.L, 12 * res + 1);
        double trsh = 0.05 * Constants.ST;
        //double[] Ynodes = GenericBlas.Linspace(0, Constants.ST - 2 * trsh, res + 1);
        //Ynodes = Ynodes.Cat(GenericBlas.Linspace(Constants.ST + trsh, 2 * Constants.L, 2 * res + 1));
        double[] Ynodes = GenericBlas.Linspace(0, 2.0 * Constants.L, 2 * res + 1);
        double[] Znodes = GenericBlas.Linspace(- Constants.L, 0.0 * Constants.L, res + 1);


        var grd = Grid3D.Cartesian3DGrid(Xnodes, Ynodes, Znodes);

        //grd.EdgeTagNames.Add(1, "freeslip_ZeroGradient");
        grd.EdgeTagNames.Add(1, "wall_ZeroGradient");
        grd.EdgeTagNames.Add(2, "pressure_outlet_ConstantTemperature");
        grd.EdgeTagNames.Add(3, "wall_ConstantTemperature");
        grd.EdgeTagNames.Add(4, "freeslip_ZeroGradient");
        
        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 1;
            if((Math.Abs(X[0] - Xnodes.First()) < 1e-6) || (Math.Abs(X[0]- Xnodes.Last()) < 1e-6))
                return 1;
            else if((Math.Abs(X[1] - Ynodes.Last()) < 1e-6) )
                return 2;
            else if((Math.Abs(X[1] - Ynodes.First()) < 1e-6) )
                return 3;
            else if((Math.Abs(X[2] - Znodes.Last()) < 1e-6) )
                return 4;
            else if((Math.Abs(X[2] - Znodes.First()) < 1e-6) )
                //return 1;
                return 4;
            return et;
        });

        myDb.Controller.DBDriver.SaveGrid(grd, myDb);

        return grd;
    }

    public static GridCommons GridConvergencePeriodic(int res, IDatabaseInfo myDb){
        //double h = Constants.L/res;
        double[] Xnodes = GenericBlas.Linspace(- Constants.L, 6.0 * Constants.L, 7 * res + 1);
        double trsh = 0.05 * Constants.ST;
        double[] Ynodes = GenericBlas.Linspace(0, Constants.ST - 2 * trsh, res + 1);
        Ynodes = Ynodes.Cat(GenericBlas.Linspace(Constants.ST + trsh, 2 * Constants.L, 2 * res + 1));
        //double[] Ynodes = GenericBlas.Linspace(0, 2 * Constants.L, 2 * res + 1);

        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes, periodicX: true);

        grd.EdgeTagNames.Add(1, "pressure_outlet_ConstantTemperature");
        grd.EdgeTagNames.Add(2, "wall_ConstantTemperature");
        
        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 1;
            if((Math.Abs(X[1] - Ynodes.Last()) < 1e-8) )
                return 1;
            else if((Math.Abs(X[1] - Ynodes.First()) < 1e-8) )
                return 2;
            return et;
        });

        myDb.Controller.DBDriver.SaveGrid(grd, myDb);

        return grd;
    }
}

In [17]:
public static class BoundaryAndInitialValueFactory {

    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
            stw.WriteLine("static class BoundaryAndInitialValues {");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double Phi(double[] X, double t){");
            stw.WriteLine($"         return 1.0 * ((X[0]-{Constants.X0}).Pow2() + (X[1]-{Constants.Y0}).Pow2() + (X[2]).Pow2() - {Constants.R}.Pow2());");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double Phi1(double[] X, double t){");
            //stw.WriteLine($"         return -(X[1] - {Constants.ST});");
            stw.WriteLine($"         return -(X[1] - {Constants.ST});");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double TemperatureB(double[] X, double t){");
            stw.WriteLine("         return " + Constants.T_sat + ";");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("}");
            return stw.ToString();
        }
    }

    static public Formula Get_TempB() {
        return new Formula("BoundaryAndInitialValues.TemperatureB", true, AdditionalPrefixCode: GetPrefixCode());
    }
    static public Formula Get_Phi() {
        return new Formula("BoundaryAndInitialValues.Phi", true, AdditionalPrefixCode: GetPrefixCode());
    }
    static public Formula Get_Phi1() {
        return new Formula("BoundaryAndInitialValues.Phi1", true, AdditionalPrefixCode: GetPrefixCode());
    }
}

## Initialize Controlfiles

First Simulations on a smaller domain, to see if it convegeces towards some particular shape.

In [18]:
//List<ZLS_Control> Controls = new List<ZLS_Control>();

In [19]:
public static HFSI_Control AlandT(int res = 2, int lvl = 2, int p = 2) {
        HFSI_Control C = new HFSI_Control(p);

        // basic database options
        // ======================
        C.DbPath      = Path.GetFullPath(BoSSSshell.WorkflowMgm.DefaultDatabase.Path);
        C.SessionName = "HFSI-Leidenfrost" + "_H" + res + "_AMR" + lvl + "_P" + p + "_Angle_3D_MK5_pressure_Soft";
        C.ProjectName = "HFSI-Leidenfrost";
        C.ProjectDescription = "Leidenfrost on HFSI";
        
        C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Degree", p));
        C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("AMR", lvl));
        C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Res", res));
        C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Convtype", "H"));


        // DG degrees
        // ==========
        //C.SetDGdegree(p);

        // Physical Parameters
        // ===================
        C.solveCoupledHeatEquation    = true;
        C.PhysicalParameters.Material = false;

        C.PhysicalParameters.IncludeConvection = true;
        C.ThermalParameters.IncludeConvection  = true;

        C.PhysicalParameters.rho_A = Constants.rho_A;
        C.PhysicalParameters.rho_B = Constants.rho_B;

        C.PhysicalParameters.mu_A = Constants.mu_A;
        C.PhysicalParameters.mu_B = Constants.mu_B;

        C.PhysicalParameters.Sigma = Constants.Sigma;

        C.ThermalParameters.rho_A = Constants.rho_A;
        C.ThermalParameters.rho_B = Constants.rho_B;
        C.ThermalParameters.rho_C = Constants.rho_A;

        C.ThermalParameters.c_A = Constants.c_A;
        C.ThermalParameters.c_B = Constants.c_B;
        C.ThermalParameters.c_C = Constants.c_A;

        C.ThermalParameters.k_A = Constants.k_A;
        C.ThermalParameters.k_B = Constants.k_B;
        C.ThermalParameters.k_C = Constants.k_A * 10.0;

        C.ThermalParameters.hVap  = Constants.hVap;
        C.ThermalParameters.T_sat = Constants.T_sat;

        C.PhysicalParameters.betaS_A = 0.0;
        C.PhysicalParameters.betaS_B = 0.0;

        C.PhysicalParameters.betaL      = 0.0;
        C.PhysicalParameters.sliplength = 0.0;
        C.PhysicalParameters.theta_e    = Math.PI / 2.0; 

        C.Material = new Solid() {
            Density = Constants.rho_A,
            Lame2 = 1e2,
            //Lame2 = 1e10,
            Viscosity = Constants.mu_A
        };

        // grid generation
        // ===============
        bool periodic = false;
        var grd = periodic ? GridFactory.GridConvergencePeriodic(res, BoSSSshell.WorkflowMgm.DefaultDatabase) : GridFactory.GridConvergence(res, BoSSSshell.WorkflowMgm.DefaultDatabase);
        C.SetGrid(grd);
        
        // Initial Values
        // ==============
        C.AddInitialValue("Phi", BoundaryAndInitialValueFactory.Get_Phi());
        C.AddInitialValue("Phi2", BoundaryAndInitialValueFactory.Get_Phi1());
        C.AddInitialValue("Temperature#A", "X => "+Constants.T_sat, false);
        C.AddInitialValue("Temperature#B", BoundaryAndInitialValueFactory.Get_TempB());
        C.AddInitialValue("Temperature#C", $"X => + {Constants.T_sat} + ({Constants.T_wall}-{Constants.T_sat}) * (1.0- X[1]/{Constants.ST})", false);
        C.AddInitialValue("GravityX#A", new Formula($"X =>  {Constants.g} * Math.Sin(Math.PI / 6)",false));
        C.AddInitialValue("GravityY#A", new Formula($"X => -{Constants.g} * Math.Cos(Math.PI / 6)",false));
        //C.AddInitialValue("GravityX#B", new Formula($"X =>  {Constants.g} * Math.Sin(Math.PI / 6)",false));
        //C.AddInitialValue("GravityY#B", new Formula($"X => -{Constants.g} * Math.Cos(Math.PI / 6)",false));
        //C.AddInitialValue("GravityY#C", new Formula($"X => -{Constants.g}",false));


        // boundary conditions
        // ===================
        if(!periodic) C.AddBoundaryValue("wall_ZeroGradient");
        C.AddBoundaryValue("freeslip_ZeroGradient");
        C.AddBoundaryValue("pressure_outlet_ConstantTemperature", "Temperature#A", "X => " + Constants.T_sat, false);
        if(Constants.ST > 0.0) {
            C.AddBoundaryValue("wall_ConstantTemperature", "Temperature#C", "X => " + Constants.T_wall, false);
        } else {
            C.AddBoundaryValue("wall_ConstantTemperature", "Temperature#B", "X => " + Constants.T_wall, false);
        }


        // level set options
        // =================
        C.Option_LevelSetEvolution                        = BoSSS.Solution.LevelSetTools.LevelSetEvolution.StokesExtension;
        C.Timestepper_LevelSetHandling                    = BoSSS.Solution.XdgTimestepping.LevelSetHandling.LieSplitting;
        C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.LevelSetTools.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;


        // Timestepping
        // ============
        C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
        C.TimeSteppingScheme = BoSSS.Solution.XdgTimestepping.TimeSteppingScheme.BDF2;
        C.dtFixed          = Constants.dt((int)(res * Math.Pow(2, lvl)), p); // Use same timestep for all, better comparability
        C.Endtime          = 5.0;
        C.NoOfTimesteps    = (int)Math.Ceiling(C.Endtime /C.dtFixed);
        C.saveperiod       = 1;
        C.rollingSaves     = false;
        
        // AMR
        // ============
        int level = lvl;
        C.AdaptiveMeshRefinement = lvl > 0;
        C.activeAMRlevelIndicators.Add(new BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater.AMRonNarrowband(){ levelSet = 0, maxRefinementLevel = level});
        //C.activeAMRlevelIndicators.Add(new BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater.AMRonNarrowband(){ levelSet = 1, maxRefinementLevel = level - 1});
        C.AMR_startUpSweeps = lvl;
        //C.activeAMRlevelIndicators.Add(new BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater.AMRatInnerContactLine { maxRefinementLevel = level, levelSets = new[] { 0, 1 }, FieldWidth = 1 });

        C.DynamicLoadBalancing_On = true;
        C.DynamicLoadBalancing_Period = 20;
        C.DynamicLoadBalancing_RedistributeAtStartup = true;
        C.GridPartType = GridPartType.METIS;

        C.ReInitPeriod = 100;

        // special options
        // ===============
        C.NonLinearSolver.MaxSolverIterations = 25;
        C.NonLinearSolver.MinSolverIterations = 3;
        C.NonLinearSolver.ConvergenceCriterion = 1e-8;
        C.FailOnSolverFail = false;
        //C.PostprocessingModules.Add(new BoSSS.Application.XNSFE_Solver.PhysicalBasedTestcases.MassfluxLogging() { LogPeriod = 1 }); // further logging?

        return C;
}

In [20]:
//DirectoryInfo NativeOverride = null;
//DirectoryInfo ManagedOverride = null;
//var bpc = BoSSSshell.GetDefaultQueue();
//string RelManagedPath = null;

//string mngdir = proj + "_" + DateTime.Now.ToString("MMMdd_HHmmss") + "_managed";
//ManagedOverride = new DirectoryInfo(Path.Combine(bpc.DeploymentBaseDirectory, mngdir));
//ManagedOverride.Create();
//typeof(XNSFE).Assembly.DeployAt(ManagedOverride);
//RelManagedPath = "../" + mngdir + "/" + Path.GetFileName(typeof(XNSFE).Assembly.Location);

//string ntvdir = proj + "_"  + DateTime.Now.ToString("MMMdd_HHmmss") + "_amd64";
//NativeOverride = new DirectoryInfo(Path.Combine(bpc.DeploymentBaseDirectory, ntvdir));
//NativeOverride.Create();
//MetaJobMgrIO.CopyDirectoryRec(ilPSP.Environment.NativeLibraryDir, NativeOverride.FullName, null);

//bpc.DeployRuntime = false; // we did this manually!

In [21]:
//NativeOverride.FullName.Display();
//ManagedOverride.FullName.Display();

In [28]:
var C_CTRLFILE = AlandT(1,3,2);//Create control file.
var job = C_CTRLFILE.CreateJob();
job.NumberOfMPIProcs = 4;
job.NumberOfThreads = 1;
job.Activate(myBatch);
//job.ShowOutput();

Grid Edge Tags changed.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job HFSI-Leidenfrost_H1_AMR3_P2_Angle_3D_MK5_pressure_Soft ... 
Deploying executables and additional files ...
Deployment directory: \\fdygitrunner\BoSSStests\Miao-Temp\deploy\HFSI-Leidenfrost-Angle-3d-HFSISolver2026Mar25_203622.231296
copied 53 files.
   written file: control.obj
deployment finished.


In [492]:
//BoSSSshell.WorkflowMgm.AllJobs.ForEach(j => j.Value.DeleteOldDeploymentsAndSessions());
//BoSSSshell.WorkflowMgm.AllJobs.Clear();

In [39]:
wmg.Sessions

#0: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR4_P2_Angle_3D_MK5_pressure*	4/13/2025 2:12:16 PM	410bbe07...
#1: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR4_P2_Angle_3D_MK5_freeslip*	4/11/2025 3:41:01 PM	4c4f2929...
#2: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR4_P2_Angle_3D_MK5*	4/10/2025 2:44:00 PM	7755c709...
#3: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR4_P2_Angle_3D_MK5_pressure_Soft_Restart_Restart*	3/29/2026 10:43:56 AM	582cc6f2...
#4: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR4_P2_Angle_3D_MK5_pressure_Soft_Restart*	3/28/2026 2:48:42 PM	8b560db1...
#5: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR3_P2_Angle_3D_MK5_pressure_Soft_Restart*	3/26/2026 11:15:35 AM	199b67e5...
#6: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR3_P2_Angle_3D_MK5_pressure_Soft*	3/25/2026 8:36:39 PM	5154ef53...
#7: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR4_P2_Angle_3D_MK5_pressure_Soft*	3/25/2026 8:16:35 PM	8c5421be...
#8: HFSI-Leidenfrost-Angle-3d	HFSI-Lei

In [23]:
wmg.Sessions[3].Timesteps.Count

242

In [ ]:
//int jj = 263;
//int kk = 177;

In [41]:
wmg.Sessions[7].Timesteps

#0:  { Time-step: 0.0; Physical time: 0s; Fields: Phi, PhiDG, Phi2, Phi2DG, VelocityX, VelocityY, VelocityZ, Pressure, Temperature, DisplacementX, DisplacementY, DisplacementZ, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Residual-HeatEq, Residual-DisplaceX, Residual-DisplaceY, Residual-DisplaceZ, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, MPIrank, CutCells; Name:  }
#1:  { Time-step: 0.1; Physical time: 0s; Fields: Phi, PhiDG, Phi2, Phi2DG, VelocityX, VelocityY, VelocityZ, Pressure, Temperature, DisplacementX, DisplacementY, DisplacementZ, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Residual-HeatEq, Residual-DisplaceX, Residual-DisplaceY, Residual-DisplaceZ, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, MPIrank, CutCells; Name:  }
#2:  { Time-step: 0.2; Physical time: 0s; Fields: Phi, PhiDG, Phi2, Phi2DG, VelocityX, VelocityY, VelocityZ, Pressure, Temperature, DisplacementX, DisplacementY, DisplacementZ, R

In [86]:
wmg.Sessions[5].Timesteps.Every(1).Skip(kk).Export().WithSupersampling(3).Do();
kk = wmg.Sessions[5].Timesteps.Count

Starting export process... Data will be written to the directory: d:\Users\miao\AppData\Local\BoSSS\plots\sessions\HFSI-Leidenfrost-Angle-3d__HFSI-Leidenfrost_H1_AMR4_P2_Angle_3D_MK5_pressure_Soft__8c5421be-bb20-48c4-8dbb-25b89a10d104


606

In [209]:
//7 * wmg.Sessions[0].GetDOF(VariableNames.DisplacementX) + wmg.Sessions[0].GetDOF(BoSSS.Solution.NSECommon.VariableNames.Pressure)

In [596]:
//BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(86400);

## Restart

In [20]:
wmg.Sessions

#0: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR4_P2_Angle_3D_MK5_pressure*	4/13/2025 2:12:16 PM	410bbe07...
#1: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR4_P2_Angle_3D_MK5_freeslip*	4/11/2025 3:41:01 PM	4c4f2929...
#2: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR4_P2_Angle_3D_MK5*	4/10/2025 2:44:00 PM	7755c709...
#3: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_Angle_3D_MK5_pressure_Soft_Noreinit*	3/29/2026 11:06:48 AM	6a75307e...
#4: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR4_P2_Angle_3D_MK5_pressure_Soft_Restart_Restart*	3/29/2026 10:43:56 AM	582cc6f2...
#5: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR4_P2_Angle_3D_MK5_pressure_Soft_Restart*	3/28/2026 2:48:42 PM	8b560db1...
#6: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR3_P2_Angle_3D_MK5_pressure_Soft_Restart*	3/26/2026 11:15:35 AM	199b67e5...
#7: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR3_P2_Angle_3D_MK5_pressure_Soft*	3/25/2026 8:36:39 PM	5154ef53...
#8: HFSI-Leidenfrost-Angle-3d	HFSI-Leid

In [21]:
wmg.Sessions[3]

HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_Angle_3D_MK5_pressure_Soft_Noreinit*	3/29/2026 11:06:48 AM	6a75307e...

In [22]:
var Sloc = wmg.Sessions[3];

In [23]:
var c3 = Sloc.CreateRestartControl() as HFSI_Control;

In [24]:
c3.GetType()

HFSISolver.HFSI_Control

In [25]:
c3.GridGuid

a64c69cb-c153-4713-b5d2-295592d9d090

In [26]:
Sloc.Timesteps.Last().GridID

520beaa1-f18f-4116-b0bd-dbdd520204f4

In [27]:
c3.GridGuid = Sloc.Timesteps.Last().GridID;

In [28]:
c3.GridGuid

520beaa1-f18f-4116-b0bd-dbdd520204f4

In [29]:
//c3.activeAMRlevelIndicators.Add(new BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater.AMRonNarrowband(){ levelSet = 0, maxRefinementLevel = 4});
//c3.AMR_startUpSweeps = 4;

In [30]:
c3.DynamicLoadBalancing_On = true;
c3.DynamicLoadBalancing_Period = 50;
c3.DynamicLoadBalancing_RedistributeAtStartup = true;
c3.GridPartType = GridPartType.METIS;
c3.ReInitPeriod = 1000;

In [31]:
//c3.SessionName = "HFSI-Leidenfrost_Angle_3D_MK5_pressure_Soft_Noreinit";

In [32]:
//double dt = 1e-4;
//c2.dtMax = dt;
//c2.dtMin = dt;
//c2.Endtime = 100;
//c2.NoOfTimesteps = 100;
//c2.saveperiod = 1;

In [33]:
var JobLocal3 = c3.CreateJob();
JobLocal3.Status

PreActivation

In [34]:
JobLocal3.NumberOfMPIProcs = 8;
JobLocal3.NumberOfThreads = 2;
JobLocal3.Activate(myBatch);
//JobLocal2.ShowOutput();

Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job HFSI-Leidenfrost_Angle_3D_MK5_pressure_Soft_Noreinit_Restart ... 
Deploying executables and additional files ...
Deployment directory: \\fdygitrunner\BoSSStests\Miao-Temp\deploy\HFSI-Leidenfrost-Angle-3d-HFSISolver2026Mar30_205341.010571
copied 53 files.
   written file: control.obj
deployment finished.
Instance #1 of ssh client miao@fdycluster instantiated.


In [55]:
wmg.Sessions

#0: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR4_P2_Angle_3D_MK5_pressure*	4/13/2025 2:12:16 PM	410bbe07...
#1: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR4_P2_Angle_3D_MK5_freeslip*	4/11/2025 3:41:01 PM	4c4f2929...
#2: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR4_P2_Angle_3D_MK5*	4/10/2025 2:44:00 PM	7755c709...
#3: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_Angle_3D_MK5_pressure_Soft_Noreinit*	3/29/2026 11:06:48 AM	6a75307e...
#4: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR4_P2_Angle_3D_MK5_pressure_Soft_Restart_Restart*	3/29/2026 10:43:56 AM	582cc6f2...
#5: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR4_P2_Angle_3D_MK5_pressure_Soft_Restart*	3/28/2026 2:48:42 PM	8b560db1...
#6: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR3_P2_Angle_3D_MK5_pressure_Soft_Restart*	3/26/2026 11:15:35 AM	199b67e5...
#7: HFSI-Leidenfrost-Angle-3d	HFSI-Leidenfrost_H1_AMR3_P2_Angle_3D_MK5_pressure_Soft*	3/25/2026 8:36:39 PM	5154ef53...
#8: HFSI-Leidenfrost-Angle-3d	HFSI-Leid

In [ ]:
//int iii = 0;
//int jjj = 0;

In [54]:
wmg.Sessions[3].Timesteps.Every(1).Skip(iii).Export().WithSupersampling(3).Do();
iii = wmg.Sessions[3].Timesteps.Count;

Starting export process... Data will be written to the directory: d:\Users\miao\AppData\Local\BoSSS\plots\sessions\HFSI-Leidenfrost-Angle-3d__HFSI-Leidenfrost_Angle_3D_MK5_pressure_Soft_Noreinit__6a75307e-1a80-43a2-b19d-03c9b2756c8c


In [49]:
wmg.Sessions[4].Timesteps.Every(1).Skip(jjj).Export().WithSupersampling(3).Do();
jjj = wmg.Sessions[4].Timesteps.Count;

Starting export process... Data will be written to the directory: d:\Users\miao\AppData\Local\BoSSS\plots\sessions\HFSI-Leidenfrost-Angle-3d__HFSI-Leidenfrost_H1_AMR4_P2_Angle_3D_MK5_pressure_Soft_Restart_Restart__582cc6f2-e1ff-41eb-a3b6-b5a79329c9e0
